# Preprocessing — Dataset 1 (clean)

Tahap ini **tidak** menghasilkan fitur. Keluarannya dua fondasi yang dipakai berkali-kali
oleh notebook training, evaluasi, dan robustness study:

1. **Manifest** (`ml/manifest_d1.csv`) — daftar isi: satu baris per file, berisi
   `filename, label, source_id, path`.
2. **Split** (`ml/split_d1_{train,val,test}.csv`) — pembagian train/val/test yang
   **dibekukan ke file**, supaya semua eksperimen memakai pembagian yang sama persis.

Fitur (log-mel / embedding) ditunda ke notebook training — representasinya akan
ditentukan oleh model transfer learning yang dipilih nanti.

## Yang dikerjakan

| Langkah | D1 |
|---|---|
| Inventaris + `source_id` | dari nama file `sound_N.wav` |
| Buang duplikat | **4 file** byte-identik di `ambulance` (EDA) |
| Fungsi audio bersih | resample 22050 · mono · peak-normalize (divalidasi, belum dipakai) |
| Split | 70/15/15, group-aware (`source_id`), stratified per kelas |

> D1 tidak punya kelas `police` dan tidak punya varian overlay (`_N_1`), jadi grouping
> di sini praktis trivial — tiap rekaman = satu source. Struktur kode tetap disamakan
> dengan D2 supaya konsisten.

In [1]:
import hashlib
import re
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------------ config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
D1_DIR = ROOT / "Dataset" / "Dataset1" / "sounds"
ML_DIR = ROOT / "ml"
ML_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["ambulance", "firetruck", "traffic"]      # D1 tanpa police

# parameter audio — samakan dengan seluruh pipeline (lihat CLAUDE.md)
SR = 22050
DURATION = 3.0
N_SAMPLES = int(SR * DURATION)                        # 66150

# proporsi split
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42

assert D1_DIR.exists(), f"folder tidak ditemukan: {D1_DIR}"
print(f"dataset : {D1_DIR}")
print(f"output  : {ML_DIR}")

dataset : D:\Coding Vscode\Siren Classification\Dataset\Dataset1\sounds
output  : D:\Coding Vscode\Siren Classification\ml


---
## 1 · Inventaris + `source_id`

Baca semua `.wav`, catat label dan hash isi (untuk deteksi duplikat). `source_id`
menandai *rekaman sumber*: prefix kelas + angka pertama pada nama file. Prefix wajib
karena penomoran D1 bersifat global (lihat CLAUDE.md) — tanpa prefix, angka bisa
bentrok antar kelas.

In [2]:
def parse_source_id(stem: str, label: str) -> str:
    """Rekaman sumber = prefix kelas + angka pertama di nama file.

    sound_57.wav (ambulance) -> 'ambulance_57'. Prefix mencegah bentrok karena
    penomoran D1 global (traffic memakai 401-600, dst).
    """
    match = re.search(r"(\d+)", stem)
    base = match.group(1) if match else stem
    return f"{label}_{base}"


def build_inventory(base_dir: Path, classes: list) -> pd.DataFrame:
    """Satu baris per file .wav: identitas + source_id + hash isi."""
    rows = []
    for label in classes:
        for f in sorted((base_dir / label).glob("*.wav")):
            rows.append({
                "filename": f.name,
                "label": label,
                "source_id": parse_source_id(f.stem, label),
                "path": str(f),
                "md5": hashlib.md5(f.read_bytes()).hexdigest(),
            })
    return pd.DataFrame(rows)


df = build_inventory(D1_DIR, CLASSES)
print(f"total file .wav   : {len(df)}")
print(f"source unik       : {df.source_id.nunique()}")
print(f"per kelas         : {df.label.value_counts().reindex(CLASSES).to_dict()}")
df.head()

total file .wav   : 600
source unik       : 600
per kelas         : {'ambulance': 200, 'firetruck': 200, 'traffic': 200}


,filename,label,source_id,path,md5
0,sound_1.wav,ambulance,ambulance_1,D:\Coding Vscode\Siren Classification\Dataset\...,b564f5ccfb25ea8fd3908204b039d7b1
1,sound_10.wav,ambulance,ambulance_10,D:\Coding Vscode\Siren Classification\Dataset\...,1151028dfe391b7786445b516aefcf08
2,sound_100.wav,ambulance,ambulance_100,D:\Coding Vscode\Siren Classification\Dataset\...,276511eae2d37ba077baeef7a23e3c25
3,sound_101.wav,ambulance,ambulance_101,D:\Coding Vscode\Siren Classification\Dataset\...,89d33c2925a5d9a36a3154795c6bd4dc
4,sound_102.wav,ambulance,ambulance_102,D:\Coding Vscode\Siren Classification\Dataset\...,e94cbf1b660db1b85e53167fa69c0436


---
## 2 · Buang duplikat

EDA menemukan 4 file di `ambulance` yang **byte-identik** dengan file lain (hash sama,
nama beda). Kalau tidak dibuang, satu rekaman yang sama bisa muncul di train *dan* test
— kebocoran yang menggelembungkan akurasi. Kita simpan kemunculan pertama, buang sisanya.

In [3]:
before = len(df)
dup_mask = df.duplicated(subset="md5", keep="first")
dropped = df[dup_mask]

print(f"file duplikat (byte-identik) : {len(dropped)}")
if len(dropped):
    display(dropped[["filename", "label", "source_id"]])

df = df[~dup_mask].reset_index(drop=True)
print(f"\nsebelum : {before}  ->  sesudah : {len(df)}")
assert df.md5.duplicated().sum() == 0, "masih ada duplikat!"

file duplikat (byte-identik) : 4


,filename,label,source_id
35,sound_130.wav,ambulance,ambulance_130
174,sound_76.wav,ambulance,ambulance_76
175,sound_77.wav,ambulance,ambulance_77
176,sound_78.wav,ambulance,ambulance_78



sebelum : 600  ->  sesudah : 596


---
## 3 · Fungsi audio bersih (validasi saja)

Definisi transformasi sinyal yang **akan** dipakai saat ekstraksi fitur nanti:
resample ke 22050 Hz, jadikan mono, peak-normalize, dan paksa panjang tepat
`N_SAMPLES`. Di sini kita **hanya memvalidasi**-nya pada beberapa file — memastikan
output-nya benar (mono, panjang pas, rentang [-1, 1]). File audio tidak ditulis ulang.

In [4]:
def load_clean_audio(path: str) -> np.ndarray:
    """Audio bersih siap-fitur: 22050 Hz, mono, peak-normalized, panjang tetap.

    Durasi D1 sudah ~3.0 s, tapi kita paksa panjang tepat N_SAMPLES supaya semua
    array identik bentuknya (beberapa file 3.02 s).
    """
    y, _ = librosa.load(path, sr=SR, mono=True)          # resample + mono
    if len(y) < N_SAMPLES:                               # pad kalau kurang
        y = np.pad(y, (0, N_SAMPLES - len(y)))
    y = y[:N_SAMPLES]                                    # crop kalau lebih
    peak = np.abs(y).max()
    return y / peak if peak > 0 else y                   # peak-normalize


# --- validasi pada 1 sampel per kelas ---
print("validasi load_clean_audio:")
for cls in CLASSES:
    p = df[df.label == cls].iloc[0].path
    y = load_clean_audio(p)
    assert y.shape == (N_SAMPLES,), f"panjang salah: {y.shape}"
    assert np.abs(y).max() <= 1.0 + 1e-6, "melebihi rentang [-1, 1]"
    print(f"  {cls:10s} -> shape={y.shape}  peak={np.abs(y).max():.3f}  ok")
print("\nsemua sampel lolos validasi.")

validasi load_clean_audio:


  ambulance  -> shape=(66150,)  peak=1.000  ok
  firetruck  -> shape=(66150,)  peak=1.000  ok
  traffic    -> shape=(66150,)  peak=1.000  ok

semua sampel lolos validasi.


---
## 4 · Split train/val/test (group-aware)

Pembagian **70/15/15**. Dua aturan yang dijaga:

- **Group-aware** (`groups=source_id`) — file dari rekaman yang sama tidak boleh
  terpecah antar split. Di D1 tiap source berisi satu file, jadi ini praktis sama
  dengan split acak, tapi kode dibuat identik dengan D2 demi konsistensi.
- **Stratified** — proporsi kelas dijaga seimbang di tiap split
  (`StratifiedGroupKFold` menangani stratifikasi + grup sekaligus).

Ditutup dengan **assert** bahwa irisan `source_id` antar split kosong.

In [5]:
from sklearn.model_selection import StratifiedGroupKFold


def stratified_group_split(df, test_size, val_size, seed):
    """Bagi df jadi train/val/test, group-aware + stratified per label.

    Dua tahap StratifiedGroupKFold: pisahkan test dulu, lalu val dari sisanya.
    Return tiga DataFrame.
    """
    def carve(frame, frac):
        # ambil ~frac sebagai 'hold', sisanya 'keep' — tanpa memecah source_id
        n_splits = max(2, round(1 / frac))
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        keep_idx, hold_idx = next(sgkf.split(frame, frame.label, groups=frame.source_id))
        return frame.iloc[keep_idx], frame.iloc[hold_idx]

    rest, test = carve(df, test_size)
    # val_size dihitung relatif terhadap sisa (rest), bukan total
    train, val = carve(rest, val_size / (1 - test_size))
    return (train.reset_index(drop=True),
            val.reset_index(drop=True),
            test.reset_index(drop=True))


train_df, val_df, test_df = stratified_group_split(df, TEST_SIZE, VAL_SIZE, SEED)

# --- verifikasi tidak ada source yang bocor antar split ---
s_tr, s_va, s_te = (set(d.source_id) for d in (train_df, val_df, test_df))
assert s_tr.isdisjoint(s_va), "source bocor: train & val"
assert s_tr.isdisjoint(s_te), "source bocor: train & test"
assert s_va.isdisjoint(s_te), "source bocor: val & test"
print("OK — tidak ada source_id yang bocor antar split.\n")

# --- ringkasan ukuran & distribusi kelas ---
summary = pd.DataFrame({
    split: d.label.value_counts().reindex(CLASSES)
    for split, d in [("train", train_df), ("val", val_df), ("test", test_df)]
})
summary.loc["TOTAL"] = summary.sum()
print(summary)
print(f"\nproporsi file : train {len(train_df)/len(df):.0%} · "
      f"val {len(val_df)/len(df):.0%} · test {len(test_df)/len(df):.0%}")

OK — tidak ada source_id yang bocor antar split.

           train  val  test
label                      
ambulance    139   29    28
firetruck    144   28    28
traffic      143   28    29
TOTAL        426   85    85

proporsi file : train 71% · val 14% · test 14%


---
## 5 · Simpan manifest + split

Manifest penuh dan tiga file split ditulis ke `ml/`. Kolom `md5` dan `path` absolut
tidak ikut disimpan — `path` disimpan relatif terhadap root project supaya CSV tetap
portabel antar mesin (path absolut Windows tidak berlaku di Kaggle).

In [6]:
def relpath(p: str) -> str:
    return str(Path(p).relative_to(ROOT)).replace("\\", "/")


def save(frame, name):
    out = frame.copy()
    out["path"] = out.path.map(relpath)
    out = out[["filename", "label", "source_id", "path"]]
    dest = ML_DIR / name
    out.to_csv(dest, index=False)
    print(f"  {name:28s} {len(out):4d} baris -> {dest}")


print("menyimpan:")
save(df, "manifest_d1.csv")
save(train_df, "split_d1_train.csv")
save(val_df, "split_d1_val.csv")
save(test_df, "split_d1_test.csv")
print("\nselesai. Dipakai oleh notebook training & robustness study.")

menyimpan:
  manifest_d1.csv               596 baris -> D:\Coding Vscode\Siren Classification\ml\manifest_d1.csv
  split_d1_train.csv            426 baris -> D:\Coding Vscode\Siren Classification\ml\split_d1_train.csv
  split_d1_val.csv               85 baris -> D:\Coding Vscode\Siren Classification\ml\split_d1_val.csv
  split_d1_test.csv              85 baris -> D:\Coding Vscode\Siren Classification\ml\split_d1_test.csv

selesai. Dipakai oleh notebook training & robustness study.


---

**Selanjutnya:** `03_preprocessing_dataset2.ipynb` melakukan hal yang sama untuk D2 —
bedanya grouping di sana **krusial** (varian overlay `_N_1`), bukan sekadar formalitas.